# OLS with Fixed Effects & Clustering: polars_reg vs fixest

This notebook verifies equivalence between `polars_reg.ols()` with absorbed fixed effects
and clustered standard errors against R's `fixest::feols()`.

**Dataset**: Grunfeld (220 obs = 10 firms x 22 years) from the `plm` package.

**Model**: `inv ~ value + capital` with firm and/or year fixed effects.

**Tests**:
1. One-way FE (firm), iid SEs
2. One-way FE (firm), HC1 robust SEs
3. One-way FE (firm), clustered by firm
4. Two-way FE (firm + year), iid SEs
5. Two-way FE (firm + year), clustered by firm
6. Two-way FE (firm + year), two-way clustering (firm + year)
7. No FE, clustered by firm (baseline comparison)

In [ ]:
import sys
import tempfile

# Ensure polars_reg is importable
sys.path.insert(0, "../..")

import polars as pl
from polars_reg import ols

# Import verification helpers
sys.path.insert(0, ".")
import r_helper
from r_helper import load_r_dataset, run_r_regression, compare, R_EXTRACT

print(f"Rscript: {r_helper.RSCRIPT}")

In [ ]:
# Load the Grunfeld dataset from R's plm package
df = load_r_dataset("Grunfeld", package="plm")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")
print(f"Dtypes: {df.dtypes}")
df.head()

In [ ]:
# Ensure firm is a string column (polars_reg FE absorption handles strings)
if df["firm"].dtype != pl.Utf8:
    df = df.with_columns(pl.col("firm").cast(pl.Utf8))

# Ensure year is also a string for FE absorption
if df["year"].dtype != pl.Utf8:
    df = df.with_columns(pl.col("year").cast(pl.Utf8))

print(f"firm dtype: {df['firm'].dtype}, unique: {df['firm'].n_unique()}")
print(f"year dtype: {df['year'].dtype}, unique: {df['year'].n_unique()}")

In [ ]:
# Save to a temp CSV for R
csv_path = tempfile.mktemp(suffix=".csv")
df.to_pandas().to_csv(csv_path, index=False)
print(f"Saved to {csv_path}")

---
## Test 1: One-way FE (firm), iid SEs

In [ ]:
# polars_reg
pr1 = ols("inv ~ value + capital | firm", data=df)
pr1.summary()

In [ ]:
# R: fixest with iid SEs
r_script_1 = f'''
library(fixest)
df <- read.csv("{csv_path}")
model <- feols(inv ~ value + capital | firm, data=df, vcov="iid")
vcov_mat <- vcov(model)
{R_EXTRACT}
'''
r1 = run_r_regression(r_script_1)
print(f"R coefs: {r1.coef}")
print(f"R SEs:   {r1.se}")
comp1 = compare(pr1, r1, rtol=2e-5, se_rtol=5e-4, label="1-way FE, iid")
comp1

---
## Test 2: One-way FE (firm), HC1 robust SEs

In [ ]:
# polars_reg
pr2 = ols("inv ~ value + capital | firm", data=df, vcov="HC1")
pr2.summary()

In [ ]:
# R: fixest with heteroskedasticity-robust SEs
r_script_2 = f'''
library(fixest)
df <- read.csv("{csv_path}")
model <- feols(inv ~ value + capital | firm, data=df, vcov="hetero")
vcov_mat <- vcov(model)
{R_EXTRACT}
'''
r2 = run_r_regression(r_script_2)
print(f"R coefs: {r2.coef}")
print(f"R SEs:   {r2.se}")
comp2 = compare(pr2, r2, rtol=2e-5, se_rtol=5e-4, label="1-way FE, HC1")
comp2

---
## Test 3: One-way FE (firm), clustered by firm

In [ ]:
# polars_reg
pr3 = ols("inv ~ value + capital | firm", data=df, cluster=["firm"])
pr3.summary()

In [ ]:
# R: fixest with firm clustering
r_script_3 = f'''
library(fixest)
df <- read.csv("{csv_path}")
model <- feols(inv ~ value + capital | firm, data=df, vcov=~firm)
vcov_mat <- vcov(model)
{R_EXTRACT}
'''
r3 = run_r_regression(r_script_3)
print(f"R coefs: {r3.coef}")
print(f"R SEs:   {r3.se}")
comp3 = compare(pr3, r3, rtol=2e-5, se_rtol=5e-4, label="1-way FE, cluster(firm)")
comp3

---
## Test 4: Two-way FE (firm + year), iid SEs

In [ ]:
# polars_reg
pr4 = ols("inv ~ value + capital | firm + year", data=df)
pr4.summary()

In [ ]:
# R: fixest with two-way FE, iid SEs
r_script_4 = f'''
library(fixest)
df <- read.csv("{csv_path}")
model <- feols(inv ~ value + capital | firm + year, data=df, vcov="iid")
vcov_mat <- vcov(model)
{R_EXTRACT}
'''
r4 = run_r_regression(r_script_4)
print(f"R coefs: {r4.coef}")
print(f"R SEs:   {r4.se}")
comp4 = compare(pr4, r4, rtol=2e-5, se_rtol=5e-4, label="2-way FE, iid")
comp4

---
## Test 5: Two-way FE (firm + year), clustered by firm

In [ ]:
# polars_reg
pr5 = ols("inv ~ value + capital | firm + year", data=df, cluster=["firm"])
pr5.summary()

In [ ]:
# R: fixest with two-way FE, cluster by firm
r_script_5 = f'''
library(fixest)
df <- read.csv("{csv_path}")
model <- feols(inv ~ value + capital | firm + year, data=df, vcov=~firm)
vcov_mat <- vcov(model)
{R_EXTRACT}
'''
r5 = run_r_regression(r_script_5)
print(f"R coefs: {r5.coef}")
print(f"R SEs:   {r5.se}")
comp5 = compare(pr5, r5, rtol=2e-5, se_rtol=5e-4, label="2-way FE, cluster(firm)")
comp5

---
## Test 6: Two-way FE (firm + year), two-way clustering (firm + year)

In [ ]:
# polars_reg
pr6 = ols("inv ~ value + capital | firm + year", data=df, cluster=["firm", "year"])
pr6.summary()

In [ ]:
# R: fixest with two-way FE, two-way clustering
r_script_6 = f'''
library(fixest)
df <- read.csv("{csv_path}")
model <- feols(inv ~ value + capital | firm + year, data=df, vcov=~firm+year)
vcov_mat <- vcov(model)
{R_EXTRACT}
'''
r6 = run_r_regression(r_script_6)
print(f"R coefs: {r6.coef}")
print(f"R SEs:   {r6.se}")
comp6 = compare(pr6, r6, rtol=2e-5, se_rtol=5e-4, label="2-way FE, cluster(firm+year)")
comp6

---
## Test 7: No FE, clustered by firm (baseline comparison)

In [ ]:
# polars_reg
pr7 = ols("inv ~ value + capital", data=df, cluster=["firm"])
pr7.summary()

In [ ]:
# R: fixest with no FE, cluster by firm
r_script_7 = f'''
library(fixest)
df <- read.csv("{csv_path}")
model <- feols(inv ~ value + capital, data=df, vcov=~firm)
vcov_mat <- vcov(model)
{R_EXTRACT}
'''
r7 = run_r_regression(r_script_7)
print(f"R coefs: {r7.coef}")
print(f"R SEs:   {r7.se}")
comp7 = compare(pr7, r7, rtol=1e-6, se_rtol=1e-6, label="No FE, cluster(firm)")
comp7

---
## Summary

All seven test configurations compare `polars_reg.ols()` against `fixest::feols()` on the Grunfeld dataset:

| # | Fixed Effects | SE Type | Coef tol | SE tol |
|---|-------------|---------|----------|--------|
| 1 | firm | iid | 2e-5 | 5e-4 |
| 2 | firm | HC1 (robust) | 2e-5 | 5e-4 |
| 3 | firm | cluster(firm) | 2e-5 | 5e-4 |
| 4 | firm + year | iid | 2e-5 | 5e-4 |
| 5 | firm + year | cluster(firm) | 2e-5 | 5e-4 |
| 6 | firm + year | cluster(firm+year) | 2e-5 | 5e-4 |
| 7 | none | cluster(firm) | 1e-6 | 1e-6 |

FE models use looser tolerance (2e-5 coef, 5e-4 SE) due to demeaning algorithm differences.
The non-FE model uses tighter tolerance (1e-6) since there is no iterative demeaning.

In [ ]:
# Clean up temp file
import os
os.unlink(csv_path)
print("Temp CSV removed.")